# Tuần 3 (Sprint S1) — Huấn luyện DistilBERT E5, Đóng băng Encoder E7 & Sinh Nhãn Mềm LLM P2 (TV3 — Tasks 3.9, 3.10, 3.11)

Notebook này thực thi các thí nghiệm GPU Tuần 3 của **TV3** trên Google Colab (`Tesla T4`) theo `docs/WEEK3.md`:
1. **Task 3.9 (`E5` — DistilBERT fine-tune toàn bộ, 3 seeds `[1, 2, 3]`):** Đã hoàn thành và tích hợp vào `results/E5/seed{1,2,3}/` (`Macro-F1 = 0.8030 ± 0.0054`).
2. **Task 3.10 (`E7` — DistilBERT đóng băng encoder, 3 seeds `[1, 2, 3]`):** Đã hoàn thành và tích hợp vào `results/E7/seed{1,2,3}/` (`Macro-F1 = 0.5913 ± 0.0023`).
3. **Task 3.11 (Sinh nhãn mềm LLM với prompt `P2` trên kho `train_unlabeled` năm 2026):**
   - Đọc `data/raw/headlines_2026/headlines.jsonl` và kiểm tra bắt buộc `SHA-256 == 28bd2f3d6c2622fb10a7b3d83399996aee1ca4e83e62efe2669e99fd4d9e988f`.
   - Tự động lọc chỉ giữ `pool == "train_unlabeled"` (`1,409` bản ghi GDELT) và loại trừ tuyệt đối toàn bộ `330` ID trong `labeling/trial30_ids.csv` (`30` ID) và `labeling/assignment.csv` (`300` ID test).
   - Chạy `Qwen/Qwen2.5-7B-Instruct` 4-bit NF4 với prompt `P2`, hỗ trợ chạy tiếp (resume) qua Google Drive nếu Colab ngắt kết nối.
   - Xuất `results/llm_soft_labels/p2_soft_labels.jsonl` (chỉ gồm ID và xác suất/log-prob, tuyệt đối không chứa văn bản tiêu đề hay URL) và `results/llm_soft_labels/run.json`.

## 1. Cài đặt Môi trường Colab, Kiểm tra GPU, Clone Nhánh `tv3/week3` & Cài đặt Dependencies

- Nạp `HF_TOKEN` và `GH_TOKEN` qua **Colab Secrets** (`google.colab.userdata`).
- Kiểm tra GPU (`Tesla T4`) và cài đặt các thư viện tương thích (`transformers`, `bitsandbytes`, `accelerate`).

In [ ]:
import base64
import os
import platform
import subprocess
import sys
from pathlib import Path
import torch
from google.colab import userdata

assert torch.cuda.is_available(), "CUDA GPU not detected! Please select Runtime -> Change runtime type -> T4 GPU."
print(f"GPU detected: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

token_val = userdata.get("HF_TOKEN")
if not token_val:
    raise RuntimeError("Missing HF_TOKEN in Colab Secrets.")
os.environ["HF_TOKEN"] = token_val.strip()
del token_val

REPO_URL = "https://github.com/thienandinh29/financial-sentiment-dl.git"
REPO_DIR = Path("financial-sentiment-dl")
BRANCH = "tv3/week3"

if not REPO_DIR.exists():
    gh_val = userdata.get("GH_TOKEN")
    if not gh_val:
        raise RuntimeError("Missing GH_TOKEN in Colab Secrets.")
    basic_auth = base64.b64encode(f"x-access-token:{gh_val.strip()}".encode("utf-8")).decode("ascii")
    git_env = os.environ.copy()
    git_env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic_auth}",
    })
    del gh_val, basic_auth
    res = subprocess.run(
        ["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        env=git_env,
        capture_output=True,
        text=True,
        check=False,
    )
    del git_env
    if res.returncode != 0:
        raise RuntimeError(f"git clone failed with exit code {res.returncode}.")

if Path.cwd().name != REPO_DIR.name and REPO_DIR.exists():
    os.chdir(REPO_DIR)

remote_origin = subprocess.run(
    ["git", "config", "--get", "remote.origin.url"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
assert remote_origin == REPO_URL, "Unexpected remote.origin.url format."

py_ver = platform.python_version()
print(f"Python version: {py_ver}")
if Path("requirements.lock").is_file():
    lock_proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.lock"],
        capture_output=True,
        text=True,
        check=False,
    )
    if lock_proc.returncode == 0:
        print("Installed dependencies from requirements.lock successfully.")
    else:
        print(f"requirements.lock failed on Python {py_ver}; falling back to requirements.txt + bitsandbytes.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "bitsandbytes>=0.43.0", "accelerate>=0.30.0"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "bitsandbytes>=0.43.0", "accelerate>=0.30.0"], check=True)

if not (Path("data/raw/fns/train.jsonl").is_file() and Path("data/raw/fns/validation.jsonl").is_file()):
    subprocess.run([sys.executable, "data/download.py", "--only", "fns"], check=True)

## 2. Task 3.9 — Chạy `E5` (DistilBERT Fine-Tune Toàn Bộ, 3 Seeds)

- *(Đã hoàn tất và tích hợp vào `results/E5/seed{1,2,3}/` trên nhánh `tv3/week3`; giữ lại cell này để tái lập khi cần)*.

In [ ]:
import json
import numpy as np
import yaml
from pathlib import Path
from evaluate.metrics import read_preds, score, validation_gold
from models.transformer import run_transformer_experiment

e5_cfg = yaml.safe_load(Path("configs/E5.yaml").read_text(encoding="utf-8"))
e5_summaries = run_transformer_experiment(e5_cfg)
gold = validation_gold(e5_cfg)

e5_f1s = []
for s in e5_summaries:
    seed = s["seed"]
    pred_path = Path(f"results/E5/seed{seed}/preds_validation.jsonl")
    scored = score(read_preds(pred_path), gold)
    e5_f1s.append(scored["macro_f1"])
    print(f"E5 seed {seed}: best_epoch={s['best_epoch']} | val_macro_f1={scored['macro_f1']:.4f} | f1={scored['f1']}")

print(f"E5 Validation Macro-F1 (3 seeds): {np.mean(e5_f1s):.4f} +/- {np.std(e5_f1s, ddof=1):.4f}")

## 3. Task 3.10 — Chạy `E7` (DistilBERT Đóng Băng Encoder, 3 Seeds)

- *(Đã hoàn tất và tích hợp vào `results/E7/seed{1,2,3}/` trên nhánh `tv3/week3`; giữ lại cell này để tái lập khi cần)*.

In [ ]:
import json
import numpy as np
import yaml
from pathlib import Path
from evaluate.metrics import read_preds, score, validation_gold
from models.transformer import run_transformer_experiment

e7_cfg = yaml.safe_load(Path("configs/E7.yaml").read_text(encoding="utf-8"))
e7_summaries = run_transformer_experiment(e7_cfg)
gold = validation_gold(e7_cfg)

e7_f1s = []
for s in e7_summaries:
    seed = s["seed"]
    pred_path = Path(f"results/E7/seed{seed}/preds_validation.jsonl")
    scored = score(read_preds(pred_path), gold)
    e7_f1s.append(scored["macro_f1"])
    print(f"E7 seed {seed}: trainable_params={s['trainable_params']}/{s['total_params']} | best_epoch={s['best_epoch']} | val_macro_f1={scored['macro_f1']:.4f} | f1={scored['f1']}")

print(f"E7 Validation Macro-F1 (3 seeds): {np.mean(e7_f1s):.4f} +/- {np.std(e7_f1s, ddof=1):.4f}")

## 4. Task 3.11 — Bước 4A: Mount Google Drive (Tùy chọn), Tải Lên & Xác Minh `headlines.jsonl`

- Có thể mount Google Drive (nếu đặt `USE_GOOGLE_DRIVE = True`) để lưu tiến độ chạy tiếp (resume) và đọc `headlines.jsonl` từ Drive, hoặc upload trực tiếp file `headlines.jsonl` từ máy cá nhân.
- Xác minh bắt buộc `SHA-256 == 28bd2f3d6c2622fb10a7b3d83399996aee1ca4e83e62efe2669e99fd4d9e988f`, kiểm tra `2,168` bản ghi (`1,409` `train_unlabeled`, `759` `test_candidate`), `330` ID loại trừ (`30` trial + `300` test), và `1,409` bản ghi hợp lệ để chạy Soft Labels.

In [ ]:
import json
import shutil
from pathlib import Path
from google.colab import drive, files
from models.llm_soft_labels import (
    DEFAULT_EXCLUSION_FILES,
    DEFAULT_HEADLINES,
    DEFAULT_OUT_DIR,
    EXPECTED_HEADLINES_SHA256,
    load_excluded_ids,
    load_p2_prompt_template,
    load_unlabeled_2026_pool,
)
from train.data import sha256_file

USE_GOOGLE_DRIVE = False  # Đổi thành True nếu muốn lưu/đọc từ Google Drive
DRIVE_WORK_DIR = Path("/content/drive/MyDrive/financial_sentiment_dl_week3")

if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    DRIVE_WORK_DIR.mkdir(parents=True, exist_ok=True)
    drive_headlines = DRIVE_WORK_DIR / "headlines.jsonl"
    if drive_headlines.is_file() and not DEFAULT_HEADLINES.is_file():
        DEFAULT_HEADLINES.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_headlines, DEFAULT_HEADLINES)
        print(f"Copied headlines.jsonl from Google Drive: {drive_headlines}")

if not DEFAULT_HEADLINES.is_file():
    DEFAULT_HEADLINES.parent.mkdir(parents=True, exist_ok=True)
    print("Please upload headlines.jsonl (1,193,922 bytes, SHA-256: 28bd2f3d...):")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file uploaded!")
    uploaded_name = next(iter(uploaded.keys()))
    shutil.move(uploaded_name, DEFAULT_HEADLINES)
    if USE_GOOGLE_DRIVE:
        shutil.copy2(DEFAULT_HEADLINES, DRIVE_WORK_DIR / "headlines.jsonl")

actual_sha = sha256_file(DEFAULT_HEADLINES)
print(f"headlines.jsonl SHA-256: {actual_sha}")
assert actual_sha == EXPECTED_HEADLINES_SHA256, f"SHA-256 mismatch! Expected {EXPECTED_HEADLINES_SHA256}, got {actual_sha}"

excluded_ids = load_excluded_ids(DEFAULT_EXCLUSION_FILES)
eligible_items, pool_stats = load_unlabeled_2026_pool(
    DEFAULT_HEADLINES,
    excluded_ids=excluded_ids,
    expected_sha256=EXPECTED_HEADLINES_SHA256,
)
prompt_template, prompt_cfg = load_p2_prompt_template()
print("Pool & Exclusion Audit:")
print(json.dumps(pool_stats, indent=2))
print(f"Loaded Prompt P2 (model={prompt_cfg['model']['name']}, quant={prompt_cfg['model'].get('quantization')})")
assert pool_stats["total_rows"] == 2168
assert pool_stats["pool_counts"] == {"test_candidate": 759, "train_unlabeled": 1409}
assert pool_stats["total_excluded_ids"] == 330
assert pool_stats["excluded_overlap_train_unlabeled"] == 0
assert pool_stats["eligible_train_unlabeled"] == 1409

## 5. Task 3.11 — Bước 4B: Chạy Qwen2.5-7B-Instruct 4-bit Sinh Nhãn Mềm `P2` (Hỗ trợ Resume)

- Chấm điểm `1,409` tiêu đề thuộc `pool == "train_unlabeled"`.
- Mỗi bản ghi được flush ngay lập tức xuống `results/llm_soft_labels/p2_soft_labels.jsonl`. Nếu chạy lại cell này sau khi bị ngắt, các ID đã chấm sẽ được bỏ qua tự động.

In [ ]:
import gc
import json
import shutil
from pathlib import Path
import torch
from models.llm_soft_labels import (
    DEFAULT_HEADLINES,
    DEFAULT_OUT_DIR,
    EXPECTED_HEADLINES_SHA256,
    generate_soft_labels,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if USE_GOOGLE_DRIVE and (DRIVE_WORK_DIR / "p2_soft_labels.jsonl").is_file() and not (DEFAULT_OUT_DIR / "p2_soft_labels.jsonl").is_file():
    DEFAULT_OUT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_WORK_DIR / "p2_soft_labels.jsonl", DEFAULT_OUT_DIR / "p2_soft_labels.jsonl")
    print("Restored partial p2_soft_labels.jsonl checkpoint from Google Drive for resume.")

soft_summary = generate_soft_labels(
    headlines_path=DEFAULT_HEADLINES,
    out_dir=DEFAULT_OUT_DIR,
    expected_sha256=EXPECTED_HEADLINES_SHA256,
)
print(json.dumps(soft_summary, indent=2))

if USE_GOOGLE_DRIVE:
    shutil.copy2(DEFAULT_OUT_DIR / "p2_soft_labels.jsonl", DRIVE_WORK_DIR / "p2_soft_labels.jsonl")
    shutil.copy2(DEFAULT_OUT_DIR / "run.json", DRIVE_WORK_DIR / "run.json")
    print(f"Backed up outputs to Google Drive: {DRIVE_WORK_DIR}")

## 6. Task 3.11 — Bước 4C: Kiểm Tra Toàn Vẹn Đầu Ra & Đóng Gói `llm_soft_labels_results.zip` Để Tải Về

- Kiểm tra `results/llm_soft_labels/p2_soft_labels.jsonl` và `results/llm_soft_labels/run.json`:
  - Đúng `1,409` ID duy nhất thuộc `train_unlabeled`.
  - `0` ID trùng với `30` trial IDs hoặc `300` test IDs.
  - Không có trường văn bản tiêu đề hay URL (`text`, `title`, `headline`, `url`).
  - Xác suất 3 lớp (`bearish, neutral, bullish`) hữu hạn, thuộc `[0, 1]`, tổng xấp xỉ `1.0` (`tolerance <= 1e-3`).
- Nén thành `llm_soft_labels_results.zip` và tải trực tiếp về máy để tích hợp vào repository.

In [ ]:
import json
import math
import zipfile
from pathlib import Path
from google.colab import files
from evaluate.metrics import read_preds
from models.llm_soft_labels import (
    DEFAULT_EXCLUSION_FILES,
    DEFAULT_OUT_DIR,
    load_excluded_ids,
    validate_soft_label_row,
)

out_jsonl = DEFAULT_OUT_DIR / "p2_soft_labels.jsonl"
out_run = DEFAULT_OUT_DIR / "run.json"
assert out_jsonl.is_file() and out_run.is_file(), "Missing soft-label output files!"

excluded_ids = load_excluded_ids(DEFAULT_EXCLUSION_FILES)
preds = read_preds(out_jsonl)
run_meta = json.loads(out_run.read_text(encoding="utf-8"))

rows = [json.loads(line) for line in out_jsonl.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(rows) == 1409, f"Expected 1409 rows, got {len(rows)}"
assert len(preds) == 1409, f"Expected 1409 unique IDs, got {len(preds)}"
assert excluded_ids.isdisjoint(preds.keys()), "LEAKAGE ERROR: trial/test ID found in output!"

max_sum_err = 0.0
for r in rows:
    validate_soft_label_row(r)
    assert "letter_logp" in r and len(r["letter_logp"]) == 3
    max_sum_err = max(max_sum_err, abs(sum(r["probs"]) - 1.0))

print(f"VERIFICATION PASSED: {len(preds)} unique soft-label predictions | max_prob_sum_err={max_sum_err:.2e} | 0 excluded IDs | 0 text fields.")
print(f"min_abc_mass={run_meta['min_abc_mass']} | mean_abc_mass={run_meta['mean_abc_mass']} | share_top1_is_letter={run_meta['share_top1_is_letter']}")

zip_path = Path("llm_soft_labels_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(out_jsonl, arcname="results/llm_soft_labels/p2_soft_labels.jsonl")
    zf.write(out_run, arcname="results/llm_soft_labels/run.json")

print(f"Created {zip_path} ({zip_path.stat().st_size} bytes). Triggering download...")
files.download(str(zip_path))